In [ ]:
!pip install requests pandas matplotlib seaborn -q

import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import time

print("Libraries imported!")

In [ ]:
API_KEY = "open router api key "

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

print("API Setup Done!")

In [ ]:
tickets = [
    "My internet connection has been down since yesterday morning",
    "I was charged twice for my monthly subscription",
    "How do I reset my account password?",
    "The app keeps crashing every time I open it",
    "I want to upgrade my current plan to premium",
    "My invoice shows incorrect amount for last month",
    "I cannot login to my account since last night",
    "The video streaming quality is very poor and keeps buffering",
    "I want to cancel my subscription immediately",
    "My router is showing red light and not connecting",
    "I need a refund for the payment made yesterday",
    "How do I add a new user to my account?",
    "The mobile app is not available in my country",
    "I have been waiting for technician for 3 days",
    "Wrong email address linked to my account",
    "My data limit was reached before month end",
    "I want to change my billing date",
    "Service has been very slow for past week",
    "I accidentally deleted my account data",
    "New device not connecting to my account"
]

df = pd.DataFrame({'ticket': tickets})
print(f"Total tickets: {len(df)}")
df.head()

In [ ]:
def zero_shot_tagging(ticket):
    prompt = f"""You are a support ticket classifier.

Classify this support ticket into TOP 3 most relevant tags.
Choose from: Technical Issue, Billing, Account, Connectivity,
App Bug, Cancellation, Upgrade, Refund, Password, Data Usage

Support Ticket: "{ticket}"

Return ONLY a JSON array with exactly 3 tags.
Example: ["Technical Issue", "Connectivity", "Urgent"]
Return nothing else."""

    payload = {
        "model": "openrouter/auto",
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers=HEADERS,
        json=payload
    )

    result = response.json()
    if 'choices' in result:
        content = result['choices'][0]['message']['content']
        try:
            tags = json.loads(content)
            return tags
        except:
            return [content]
    return ["Error"]

print("Zero-shot function ready!")

In [ ]:
def few_shot_tagging(ticket):
    prompt = f"""You are a support ticket classifier.

Here are some examples:

Ticket: "Internet not working since morning"
Tags: ["Technical Issue", "Connectivity", "Urgent"]

Ticket: "Charged twice this month"
Tags: ["Billing", "Refund", "Account"]

Ticket: "Cannot login to account"
Tags: ["Account", "Password", "Technical Issue"]

Ticket: "App crashes on startup"
Tags: ["App Bug", "Technical Issue", "Urgent"]

Now classify this ticket into TOP 3 tags:
Ticket: "{ticket}"

Return ONLY a JSON array with exactly 3 tags.
Return nothing else."""

    payload = {
        "model": "openrouter/auto",
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers=HEADERS,
        json=payload
    )

    result = response.json()
    if 'choices' in result:
        content = result['choices'][0]['message']['content']
        try:
            tags = json.loads(content)
            return tags
        except:
            return [content]
    return ["Error"]

print("Few-shot function ready!")

In [ ]:
print("Running Zero-Shot and Few-Shot tagging...")
print("This will take a few minutes...\n")

zero_shot_results = []
few_shot_results = []

for i, ticket in enumerate(tickets[:10]):
    print(f"Processing ticket {i+1}/10...")

    # Zero-shot
    z_tags = zero_shot_tagging(ticket)
    zero_shot_results.append(z_tags)
    time.sleep(2)

    # Few-shot
    f_tags = few_shot_tagging(ticket)
    few_shot_results.append(f_tags)
    time.sleep(2)

df_results = df[:10].copy()
df_results['zero_shot_tags'] = zero_shot_results
df_results['few_shot_tags'] = few_shot_results

print("\nTagging complete!")
df_results

In [ ]:
print("ZERO-SHOT vs FEW-SHOT COMPARISON")
print("=" * 70)

for i, row in df_results.iterrows():
    print(f"\nTicket: {row['ticket'][:60]}...")
    print(f"Zero-Shot: {row['zero_shot_tags']}")
    print(f"Few-Shot:  {row['few_shot_tags']}")
    print("-" * 70)

In [ ]:
# Count all tags
all_tags = []
for tags in df_results['few_shot_tags']:
    if isinstance(tags, list):
        all_tags.extend(tags)

tag_counts = pd.Series(all_tags).value_counts()

plt.figure(figsize=(10,6))
sns.barplot(x=tag_counts.values, y=tag_counts.index, palette='Blues_r')
plt.title('Most Common Support Ticket Tags (Few-Shot)')
plt.xlabel('Count')
plt.ylabel('Tag')
plt.tight_layout()
plt.show()

In [ ]:
# Test your own ticket
def tag_ticket(ticket_text):
    print(f"Ticket: {ticket_text}")
    print("\nZero-Shot Tags:", zero_shot_tagging(ticket_text))
    time.sleep(2)
    print("Few-Shot Tags:", few_shot_tagging(ticket_text))

# Test examples
tag_ticket("My phone bill is higher than usual this month")